In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from tqdm import tqdm
from nba import NBAbase, NBAetl, NBAdata, NBAmodels
from betting import funcs
odds = funcs.odds()
import pickle
import shap
from nba_api.stats.endpoints import LeagueDashPtStats

In [89]:
etl = NBAetl.etl()
data = NBAdata.data()
threes = NBAmodels.models('threes')

In [94]:
df = pd.read_sql("select * from pgames where game_date = '2025-10-21'",etl.conn)

In [98]:
td = data.clean_na(threes.data)


In [101]:
qdf = pd.read_sql('''


-- This primiarily calculates rolling fields and game information for the days game to be used in the model.
-- Need to check game info in the future to see if we can preload a schedule
-- Shot attempts will be done in seperate queries as they are going through a python function for the rolling information
-- Future updates - how to incorporate inactives, can do a rolling value but would like more information on who is out and impact

SELECT 
--y
threesMade,

--identifiers
name, player_id, game_id, game_date, season, team,

--demo data
height, exp,

--shot locations, will have percentiles done in pandas
ra_fga, paint_fga, mid_fga, (COALESCE(lc_fga,0) + COALESCE(rc_fga,0)) crn_fga, abv_fga,
--minFirst, crnFgaFirst, abvFgaFirst,

--games info
CASE WHEN daysBetweenGames > 9 THEN 10 ELSE daysBetweenGames END AS daysBetweenGames,
gamesInFive, gamesInThree, oppGamesFive, OppGamesThree,
CASE WHEN oppDaysLastGame > 9 THEN 10 ELSE oppDaysLastGame END AS oppDaysLastGame,
CASE WHEN daysBetweenGames > 9  THEN 10 ELSE daysBetweenGames END -
CASE WHEN oppDaysLastGame > 9 THEN 10 ELSE oppDaysLastGame END AS netRest,

 home, tmGameCt, starter,
--rolling offensive (5 games and season) metrics
AVG(starter) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 11 PRECEDING AND 1 PRECEDING) AS mvAvgstart, 
AVG(threesMade) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) AS mvAvgThrees,
AVG(usagePercentage) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) AS mvAvgUsage,
AVG(offensiveRating) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) AS mvAvgOffRating,
AVG(marginOffRating) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) as mvAvgMarginOffRating,
AVG(min) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) as mvAvgMinutes,

    
SUM(ftm) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) * 1.0
/ SUM(fta) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) as mvAvgFtPrct,

SUM(lc_fgm+rc_fgm + abv_fgm) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) * 1.0
/    SUM(lc_fga + rc_fga + abv_fga) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) as mvAvgThrPtPrct,    

AVG(usagePercentage) OVER (PARTITION BY season,player_id
    ORDER BY game_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS seasonUsage,
AVG(offensiveRating) OVER (PARTITION BY season,player_id
    ORDER BY game_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS seasonOffRating,
    
SUM(ftm) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) * 1.0
/ SUM(fta) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as seasonFtPrct,
SUM(lc_fgm+rc_fgm + abv_fgm) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) * 1.0
/    SUM(lc_fga + rc_fga + abv_fga) OVER (PARTITION BY season,player_id
    ORDER BY plyrGameCt ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as seasonThrPtPrct,

--career metrics
SUM(ftm) OVER (PARTITION BY player_id
    ORDER BY plyrGameCt ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) * 1.0
/ SUM(fta) OVER (PARTITION BY player_id
    ORDER BY plyrGameCt ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)  as careerFtPrct,
    
SUM(lc_fgm+rc_fgm + abv_fgm) OVER (PARTITION BY player_id
    ORDER BY plyrGameCt ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) * 1.0
/    SUM(lc_fga + rc_fga + abv_fga) OVER (PARTITION BY player_id
    ORDER BY plyrGameCt ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as careerThrPtPrct, 

AVG(usagePercentage) OVER (PARTITION BY player_id
    ORDER BY game_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS careerUsage,
AVG(offensiveRating) OVER (PARTITION BY player_id
    ORDER BY game_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS careerOffRating,
AVG(threesMade) OVER (PARTITION BY player_id
    ORDER BY game_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS careerAvgThrees,
    

-- defensive information
opp_id, 
--moving (5 games) and season averages
mvAvgOppPace, mvAvgOppOpen3, mvAvgOppWide3, mvAvgOppDefrating, 
seasonOppPace,  seasonOppOpen3,  seasonOppWide3,  seasonOppDefRating


FROM pgames
WHERE player_id in 
                    (SELECT player_id
                    FROM pgames 

                    group by player_id,season

                    HAVING (AVG(min) > 15
                    and max(plyrGameCt) >= 16)

    )

order by game_date

''',etl.conn)

In [106]:
predData = data.threes_pipe(qdf[qdf.game_date=='2025-10-21'])

/opt/anaconda3/lib/python3.11/site-packages/pandas/core/window/rolling.py:609: RuntimeWarning: All-NaN slice encountered
  return func(x, start, end, min_periods, *numba_args)
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/window/rolling.py:609: RuntimeWarning: All-NaN slice encountered
  return func(x, start, end, min_periods, *numba_args)
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/window/rolling.py:609: RuntimeWarning: All-NaN slice encountered
  return func(x, start, end, min_periods, *numba_args)
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/window/rolling.py:609: RuntimeWarning: All-NaN slice encountered
  return func(x, start, end, min_periods, *numba_args)
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/window/rolling.py:609: RuntimeWarning: All-NaN slice encountered
  return func(x, start, end, min_periods, *numba_args)
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/window/rolling.py:609: RuntimeWarning: All-NaN slice encountered

In [109]:
final = data.clean_na(predData)

In [110]:
final

,threesMade,name,player_id,game_id,game_date,season,team,height,exp,daysBetweenGames,...,crn_kurtSkew,crn_fgaMv,abv_fgaMv,threesMadeMv,cornerRatio,crn_interaction,abv_interaction,wide_interaction,crResid_interaction,usage_interaction
0,0,Kevon Looney,1626172,0022500002,2025-10-21,0,GSW,81.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,Christian Wood,1626174,0022500002,2025-10-21,0,LAL,80.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,Buddy Hield,1627741,0022500002,2025-10-21,0,GSW,76.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,Gary Payton II,1627780,0022500002,2025-10-21,0,GSW,74.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,Fred VanVleet,1627832,0022500001,2025-10-21,0,HOU,72.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0,Alex Caruso,1627936,0022500001,2025-10-21,0,OKC,77.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0,Isaiah Hartenstein,1628392,0022500001,2025-10-21,0,OKC,84.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0,Dillon Brooks,1628415,0022500001,2025-10-21,0,HOU,78.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0,Shai Gilgeous-Alexander,1628983,0022500001,2025-10-21,0,OKC,78.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0,Aaron Holiday,1628988,0022500001,2025-10-21,0,HOU,72.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [116]:
threes.model.params.index[1:]

Index(['cornerRatio', 'abv_interaction', 'crn_interaction', 'netRest',
       'mvAvgThrees', 'mvAvgOppWide3', 'crn_fgaMv', 'abv_fgaMv',
       'threes_residualsAllowedMv', 'careerThrPtPrct'],
      dtype='object')

In [117]:
threes.model.predict(sm.add_constant(final.filter(threes.model.params.index[1:])))

,0,1,2,3,4,5,6,7,8,9,10
0,0.277980,0.321992,0.214410,0.106633,0.051219,0.016801,0.007049,0.002937,0.000744,0.000200,0.000036
1,0.237960,0.303130,0.241227,0.125196,0.060788,0.019050,0.008378,0.003121,0.000815,0.000288,0.000047
2,0.160569,0.278682,0.278528,0.155000,0.083747,0.024837,0.011884,0.004852,0.001138,0.000635,0.000130
3,0.278829,0.311982,0.220095,0.110395,0.051513,0.016631,0.007025,0.002601,0.000700,0.000198,0.000031
4,0.179317,0.284337,0.270860,0.148231,0.077335,0.023193,0.010867,0.004217,0.001031,0.000515,0.000096
5,0.260353,0.304480,0.232012,0.118569,0.055672,0.017623,0.007600,0.002692,0.000732,0.000233,0.000035
6,0.337191,0.305277,0.198605,0.096376,0.041164,0.013529,0.005474,0.001727,0.000523,0.000118,0.000014
7,0.213922,0.296266,0.253607,0.134426,0.067026,0.020632,0.009303,0.003501,0.000894,0.000362,0.000062
8,0.242791,0.302286,0.239984,0.124262,0.059683,0.018708,0.008199,0.002978,0.000790,0.000274,0.000044
9,0.262038,0.304741,0.231196,0.117997,0.055299,0.017523,0.007546,0.002669,0.000727,0.000230,0.000035
